# `svdynamics`: Visualizing composite SVM kernels

A reproducible, executable companion to the `svdynamics` examples. This notebook explores **decision boundaries**, **adaptive kernel weights and alignment**, **3D decision-function surfaces**, and **Gram matrix structure**.

**Important:** Adaptive weights are learned from *training labels* using uncentered kernel-target alignment. They are not automatically optimal for predictive performance. These visualizations illustrate model behavior, not unbiased performance estimates.

Run from the repository root after `pip install -e .` and `pip install matplotlib`. Figures are saved under `data_output/` at 300 DPI.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from sklearn.datasets import make_moons, make_circles, make_classification
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score
from sklearn.svm import SVC
from sklearn.metrics.pairwise import linear_kernel, rbf_kernel, polynomial_kernel
from svdynamics import CompositeKernel, AdaptiveCompositeKernel, SVDClassifier

SEED = 42
OUT = Path('data_output')
OUT.mkdir(exist_ok=True, parents=True)
SPECS = ['linear', ('rbf', {'gamma': 1.0}), ('poly', {'degree': 2, 'coef0': 1})]
LABELS = ['Linear', 'RBF', 'Polynomial']

def model_specs():
    # Fresh estimators for every dataset to avoid state reuse.
    return {
        'Linear': SVC(kernel='linear'),
        'RBF': SVC(kernel='rbf', gamma=1.0),
        'Polynomial': SVC(kernel='poly', degree=2, coef0=1),
        'Fixed composite': SVDClassifier(kernel=CompositeKernel(SPECS, weights=[0.2, 0.6, 0.2])),
        'Adaptive composite': SVDClassifier(kernel=AdaptiveCompositeKernel(SPECS)),
    }

def save(fig, filename):
    path = OUT / filename
    fig.savefig(path, dpi=300, bbox_inches='tight')
    print('Saved:', path)
    plt.show()
    plt.close(fig)


## 1. Decision boundaries: Moons and Circles

Each panel shows the **decision function**, with the black contour at zero. The plot uses the same training observations for all models to isolate geometric differences. These are **in-sample illustrations**, not estimates of generalization.

In [ ]:
datasets = {
    'Moons': make_moons(n_samples=250, noise=0.20, random_state=SEED),
    'Circles': make_circles(n_samples=250, noise=0.10, factor=0.45, random_state=SEED),
}
fig, axes = plt.subplots(2, 5, figsize=(19, 7), constrained_layout=True)
for row, (ds_name, (X, y)) in enumerate(datasets.items()):
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min()-0.5, X[:, 0].max()+0.5, 110),
        np.linspace(X[:, 1].min()-0.5, X[:, 1].max()+0.5, 110))
    grid = np.c_[xx.ravel(), yy.ravel()]
    for col, (name, estimator) in enumerate(model_specs().items()):
        pipe = make_pipeline(StandardScaler(), estimator)
        pipe.fit(X, y)
        scores = pipe.decision_function(grid).reshape(xx.shape)
        ax = axes[row, col]
        ax.contourf(xx, yy, scores, levels=24, cmap='RdBu_r', alpha=0.70)
        ax.contour(xx, yy, scores, levels=[0], colors='black', linewidths=1.5)
        ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=12,
                   edgecolor='k', linewidth=0.15)
        ax.set_title(f'{ds_name}: {name}')
        ax.set_xlabel('Feature 1')
        if col == 0: ax.set_ylabel('Feature 2')
save(fig, 'svdynamics_decision_boundaries.png')

## 2. Adaptive kernel weights and alignment

The adaptive kernel computes alignment for each candidate Gram matrix against the binary target kernel. Negative alignments are clipped to zero before normalizing weights. A zero weight indicates **no positive alignment in that particular training sample**, not a universal verdict on the kernel.

For transparency, we display both **learned weights** and **raw alignment values**.

In [ ]:
weight_datasets = {
    'Moons': make_moons(n_samples=250, noise=0.20, random_state=SEED),
    'Circles': make_circles(n_samples=250, noise=0.10, factor=0.45, random_state=SEED),
    'Linear-ish': make_classification(n_samples=250, n_features=2, n_informative=2,
                       n_redundant=0, n_clusters_per_class=1, random_state=SEED),
}
weights, alignments = [], []
for name, (X, y) in weight_datasets.items():
    Xs = StandardScaler().fit_transform(X)
    learned = AdaptiveCompositeKernel(SPECS).fit(Xs, y)
    weights.append(learned.weights_.copy())
    alignments.append(learned.alignments_.copy())
    print(f'{name:12} weights={np.round(learned.weights_, 3)} '
          f'alignments={np.round(learned.alignments_, 3)}')
weights, alignments = np.asarray(weights), np.asarray(alignments)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
x = np.arange(len(weight_datasets))
bottom = np.zeros(len(x))
for j, label in enumerate(LABELS):
    axes[0].bar(x, weights[:, j], bottom=bottom, label=label)
    bottom += weights[:, j]
axes[0].set(xticks=x, xticklabels=list(weight_datasets), ylim=(0, 1),
            ylabel='Mixture weight', title='Adaptive mixture weights')
axes[0].legend()
w = 0.24
for j, label in enumerate(LABELS):
    axes[1].bar(x+(j-1)*w, alignments[:, j], width=w, label=label)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set(xticks=x, xticklabels=list(weight_datasets),
            ylabel='Uncentered kernel-target alignment', title='Raw alignments')
axes[1].legend()
save(fig, 'svdynamics_kernel_weights_alignments.png')

## 3. Three-dimensional decision-function surfaces

The height is the **signed decision function** `f(x)`, not a probability. The horizontal plane at zero corresponds to the decision boundary. The plots use a standardized-feature grid for easier geometric interpretation and clip only the **displayed** height to prevent extreme values from obscuring the shape.

In [ ]:
X, y = make_moons(n_samples=220, noise=0.20, random_state=SEED)
scaler = StandardScaler()
Xs = scaler.fit_transform(X)
g0 = np.linspace(Xs[:, 0].min()-0.7, Xs[:, 0].max()+0.7, 65)
g1 = np.linspace(Xs[:, 1].min()-0.7, Xs[:, 1].max()+0.7, 65)
xx, yy = np.meshgrid(g0, g1)
grid = np.c_[xx.ravel(), yy.ravel()]
selected = {k: v for k, v in model_specs().items()
            if k in ('Linear', 'RBF', 'Fixed composite', 'Adaptive composite')}
fig = plt.figure(figsize=(15, 12))
for i, (name, estimator) in enumerate(selected.items(), 1):
    estimator.fit(Xs, y)
    raw = estimator.decision_function(grid).reshape(xx.shape)
    z = np.clip(raw, -2.5, 2.5)  # Display-only clipping.
    ax = fig.add_subplot(2, 2, i, projection='3d')
    ax.plot_surface(xx, yy, z, cmap='coolwarm', alpha=0.88,
                    linewidth=0, antialiased=True, rstride=2, cstride=2)
    ax.contour(xx, yy, raw, levels=[0], zdir='z', offset=-2.5,
               colors='black', linewidths=1.5)
    ax.set(xlabel='Scaled feature 1', ylabel='Scaled feature 2',
           zlabel='Decision score (clipped)', zlim=(-2.5, 2.5), title=name)
fig.suptitle('SVM decision-function geometry (Moons)', fontsize=15)
fig.tight_layout()
save(fig, 'svdynamics_3d_decision_surfaces.png')

## 4. Gram matrix heatmaps

A Gram matrix contains pairwise kernel similarities for the training observations. Here, rows and columns are **sorted by class** to reveal block structure. All three base matrices are normalized using the same training-derived scalar convention as `CompositeKernel(normalize=True)`. The final panel shows their weighted sum using weights learned from these training labels.

A heatmap alone does not establish positive semidefiniteness. For numerical PSD diagnostics, use `kernel_psd_diagnostics.py`.

In [ ]:
X, y = make_circles(n_samples=100, noise=0.10, factor=0.45, random_state=SEED)
Xs = StandardScaler().fit_transform(X)
learned = AdaptiveCompositeKernel(SPECS).fit(Xs, y)
K_base = [linear_kernel(Xs), rbf_kernel(Xs, gamma=1.0),
          polynomial_kernel(Xs, degree=2, coef0=1)]
# Use training-derived normalization scales from the fitted adaptive kernel.
K_scaled = [K / scale for K, scale in zip(K_base, learned.scales_)]
K_mix = sum(weight * K for weight, K in zip(learned.weights_, K_scaled))
# Verify the constructed mixture agrees with the library implementation.
np.testing.assert_allclose(K_mix, learned(Xs, Xs), rtol=1e-8, atol=1e-8)
order = np.argsort(y, kind='stable')
matrices = K_scaled + [K_mix]
names = LABELS + ['Learned composite']
fig, axes = plt.subplots(1, 4, figsize=(17, 4), constrained_layout=True)
for ax, name, K in zip(axes, names, matrices):
    K_sorted = K[np.ix_(order, order)]
    lim = max(np.max(np.abs(K_sorted)), 1e-10)
    im = ax.imshow(K_sorted, cmap='RdBu_r', vmin=-lim, vmax=lim,
                   interpolation='nearest', aspect='equal')
    ax.axvline(np.sum(y == 0)-0.5, color='black', linewidth=1)
    ax.axhline(np.sum(y == 0)-0.5, color='black', linewidth=1)
    ax.set_title(name)
    ax.set_xlabel('Samples (sorted by class)')
    fig.colorbar(im, ax=ax, shrink=0.72)
axes[0].set_ylabel('Samples (sorted by class)')
print('Learned weights:', dict(zip(LABELS, np.round(learned.weights_, 3))))
save(fig, 'svdynamics_gram_matrix_heatmaps.png')

## 5. Optional held-out check

The plots above explain *what the fitted models do*. To avoid conflating appearance with predictive accuracy, this final example evaluates the same kernels on a held-out test set. It is a single split, **not** a claim of statistical superiority.

In [ ]:
X, y = make_moons(n_samples=500, noise=0.25, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)
for name, estimator in model_specs().items():
    pipe = make_pipeline(StandardScaler(), estimator)
    pipe.fit(X_train, y_train)
    score = balanced_accuracy_score(y_test, pipe.predict(X_test))
    print(f'{name:20} held-out balanced accuracy = {score:.3f}')

### Exported figures

- `data_output/svdynamics_decision_boundaries.png`
- `data_output/svdynamics_kernel_weights_alignments.png`
- `data_output/svdynamics_3d_decision_surfaces.png`
- `data_output/svdynamics_gram_matrix_heatmaps.png`

**Reproducibility:** fixed random seed, train-only learning for held-out evaluation, and no use of test labels to learn adaptive weights.